In [ ]:
from google.colab import drive
from pathlib import Path
import sys
!pip install datasets transformers huggingface_hub
from datasets import load_dataset
from utils import data_structure

drive.mount('/content/drive/')
strPath = "drive/MyDrive/Spurious_Correlations/Identifying-and-Mitigating-Spurious-Correlations-for-Improving-Robustness-in-NLP-Models"
# sys.path.insert(0,"drive/MyDrive/Spurious_Correlations/Identifying-and-Mitigating-Spurious-Correlations-for-Improving-Robustness-in-NLP-Models")
filePath = Path(strPath)
%cd $filePath


Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
[Errno 2] No such file or directory: 'drive/MyDrive/Spurious_Correlations/Identifying-and-Mitigating-Spurious-Correlations-for-Improving-Robustness-in-NLP-Models'
/content/drive/MyDrive/Spurious_Correlations/Identifying-and-Mitigating-Spurious-Correlations-for-Improving-Robustness-in-NLP-Models


In [ ]:
import pandas as pd

In [ ]:
import pickle
import os
import numpy as np
import operator
dataset = "sst2"

# work_dir = "./naacl_2022"
# train_df = pd.read_csv(os.path.join(work_dir + "/transformers/glue_data/SST-2", "dev.tsv"), sep = '\t')

if dataset == "sst2":
  # train_df = pd.read_csv(os.path.join("transformers/glue_data/SST-2", "dev.tsv"), sep = '\t')
  train_df = data_structure.get_sst2()
  sentences = train_df["text"]
  labels = train_df["label"]
else:
  train_df = pd.read_csv(os.path.join("transformers/yelp_data/", "train.tsv"), sep = '\t')
  sentences = train_df["text"]
  labels = train_df["label"]


  0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
df = data_structure.get_sst2()
print(df.text)

  0%|          | 0/3 [00:00<?, ?it/s]

0             hide new secretions from the parental units 
1                     contains no wit , only labored gags 
2        that loves its characters and communicates som...
3        remains utterly satisfied to remain the same t...
4        on the worst revenge-of-the-nerds clichés the ...
                               ...                        
67344                                 a delightful comedy 
67345                     anguish , anger and frustration 
67346    at achieving the modest , crowd-pleasing goals...
67347                                    a patient viewer 
67348    this new jangle of noise , mayhem and stupidit...
Name: text, Length: 67349, dtype: object


In [ ]:
# load tokenizer and build vocab
from transformers import AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained("./models/SST_base_cased")

if dataset == "sst2":
  tokenizer = AutoTokenizer.from_pretrained("models/distilbert_SST_base_cased/") # make sure that the specified file has a config.json file in it
  vocab_filename = "models/distilbert_SST_base_cased/vocab.txt"
else:
  tokenizer = AutoTokenizer.from_pretrained("models/distilbert_yelp_base_uncased/") # make sure that the specified file has a config.json file in it
  vocab_filename = "models/distilbert_yelp_base_uncased/vocab.txt"

word2id = dict()
id2word = dict()
with open(vocab_filename, "r") as f:
    lines = f.readlines()
for idx, line in enumerate(lines):
    word2id[line.strip()] = idx
    id2word[idx] = line.strip()

In [ ]:
print(word2id['[CLS]'])
print(tokenizer("I have a apple"))

101
{'input_ids': [101, 1045, 2031, 1037, 6207, 102], 'attention_mask': [1, 1, 1, 1, 1, 1]}


### Test on SST test set

In [ ]:
# Load attention weights and predictions
if dataset == "sst2":
  attention_sst_sst = np.load("models/distilbert_SST_base_cased/cls_attention_weights.npy")
  preds = np.load("models/distilbert_SST_base_cased/predictions.npy")
else:
  attention_sst_sst = np.load("models/distilbert_yelp_base_uncased/cls_attention_weights.npy")
  preds = np.load("models/distilbert_yelp_base_uncased/predictions.npy")

In [ ]:
assert len(attention_sst_sst) == len(labels)

AssertionError: ignored

In [ ]:
print(sentences[0], labels[0])

hide new secretions from the parental units  0


In [ ]:
preds_labels = []
for elem in preds:
    if elem[0] > elem[1]:
        preds_labels.append(0)
    else:
        preds_labels.append(1)

In [ ]:
# compute acc
np.sum(np.equal(np.array(preds_labels), np.array(labels.tolist())))/len(preds_labels)

ValueError: ignored

In [ ]:
wrong_ids = []
correct_ids = []
for idx in range(len(preds_labels)):
    if preds_labels[idx] != labels[idx]:
#     if preds_labels[idx] == 0:
        wrong_ids.append(idx)
    else:
        correct_ids.append(idx)
print(len(wrong_ids))
print(len(correct_ids))

78
794


### Analysis on wrong predictions

In [ ]:
def combine_tokens(tokens, scores=None):
    if scores is None:
        token_stack = []

        if len(tokens) > 128:
            sep = tokens[-1]
            tokens = tokens[:127]
            tokens.append(sep)

        for idx, token in enumerate(tokens):

            if len(token_stack) > 0 and token_stack[-1] == "'" and token == 't':
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token
            elif len(token_stack) > 0 and token == '##s' and token_stack[-1] == "her":
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token.replace("#", "")
            elif len(token_stack) > 0 and token == '.' and token_stack[-1] in ['mr', 'mrs', 'ms']:
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token
            elif len(token_stack) > 0 and token == '-':
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token
            elif len(token_stack) > 0 and token_stack[-1][-1] == '-':
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token
            elif len(token_stack) > 0 and "'" in token:
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token
            elif "#" in token:
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token.replace("#", "")
            else:
                token_stack.append(token)
                
        return token_stack

    token_stack = []
    score_stack = []
    
    if len(tokens) > 128:
        sep = tokens[-1]
        tokens = tokens[:127]
        tokens.append(sep)
        
    for idx, token in enumerate(tokens):
                
        if len(token_stack) > 0 and token_stack[-1] == "'" and token == 't':
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token
            score_stack[-1] = new_score
        elif len(token_stack) > 0 and token == '##s' and token_stack[-1] == "her":
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token.replace("#", "")
            score_stack[-1] = new_score
        elif len(token_stack) > 0 and token == '.' and token_stack[-1] in ['mr', 'mrs', 'ms']:
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token
            score_stack[-1] = new_score
        elif len(token_stack) > 0 and token == '-':
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token
            score_stack[-1] = new_score
        elif len(token_stack) > 0 and token_stack[-1][-1] == '-':
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token
            score_stack[-1] = new_score
        elif len(token_stack) > 0 and "'" in token:
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token
            score_stack[-1] = new_score
        elif "#" in token:
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token.replace("#", "")
            score_stack[-1] = new_score
        else:
            token_stack.append(token)
            score_stack.append(scores[idx])
    assert len(token_stack) == len(score_stack)
    return token_stack, score_stack


In [ ]:
# Create a dict that contains the frequency of a token
frequency_dict = dict()
for idx in range(len(attention_sst_sst)):
    tokens = [id2word[item] for item in tokenizer(sentences[idx])["input_ids"]]
    tokens, scores = combine_tokens(tokens, attention_sst_sst[idx, :])
    for token in tokens:
        fre = frequency_dict.get(token, 0)
        frequency_dict[token] = fre + 1

In [ ]:
# most important words for neg/pos prediction
atten_words = dict()

for idx in range(len(attention_sst_sst)):
    atten = attention_sst_sst[idx, :]
    logits = preds[idx]
    logits = np.exp(logits)/sum(np.exp(logits))
    tokens = [id2word[item] for item in tokenizer(sentences[idx])["input_ids"]]
    tokens, scores = combine_tokens(tokens, atten)
    
    if logits[1] > logits[0]:
        for score_idx, score in enumerate(scores):
            token = tokens[score_idx]
            if "[SEP]" in token or "[CLS]" in token:
                continue
            
            if not token in atten_words:
                atten_words[token] = []
            atten_words[token].append(score*logits[1])
    else:
        for score_idx, score in enumerate(scores):
            token = tokens[score_idx]
            if "[SEP]" in token or "[CLS]" in token:
                continue
            
            if not token in atten_words:
                atten_words[token] = []
            atten_words[token].append(-score*logits[0])          

In [ ]:
print(len(atten_words), len(frequency_dict))
print(atten_words['lacks'])
print(frequency_dict['lacks'] == len(atten_words['lacks']))


2719 2729
[0.01993039, -0.03193205]
True


In [ ]:
use_logistic_regression = True

if not use_logistic_regression:
  new_words = dict()
  for token in atten_words:
      if len(atten_words[token]) > 0:
          # average attention for a given token
          new_words[token] = abs(sum(atten_words[token]))/len(atten_words[token])
          
  normed_words = dict()
  words_sum = sum(new_words.values())
  for k in new_words:
      normed_words[k] = np.log(new_words[k]/words_sum) - 4/np.log(1+frequency_dict[k]) # change lambda, currently set as 4
  import operator
  sorted_words = sorted(normed_words.items(), key=operator.itemgetter(1), reverse=True)

  # Token, Importance score, Frequency, average attention score
  for item in sorted_words[:50]:
      print("{: <16}".format(item[0]), "\t%.6f"%(item[1]), "\t%5d"%(len(atten_words[item[0]])), 
          "\t%.6f"%(new_words[item[0]])) # "\t{}".format("\t".join([str(item) for item in atten_words[item[0]]])))

else:
  import csv
  important_toks = []
  normed_words = {}
  frequency_dict = {}
  original_score = {}
  new_words = {}
  words_sum = 0
  if dataset == "sst2":
    with open("../emnlp-2020-spurious/tokens/sst2_important_tokens.csv", mode='r') as file:
      csvFile = csv.reader(file)
      for lines in csvFile:
        important_toks.append(lines)
        words_sum += abs(float(lines[1]))
  elif dataset == "yelp":
    with open("../emnlp-2020-spurious/tokens/yelp_important_tokens.csv", mode='r') as file:
      csvFile = csv.reader(file)
      for lines in csvFile:
        important_toks.append(lines)
        words_sum += abs(float(lines[1]))

  # print(words_sum)
  for line in important_toks:
    curr_tok = line[0]
    curr_score = float(line[1] )
    curr_freq = int(line[2])

    new_words[curr_tok] = abs(curr_score)
    frequency_dict[curr_tok] = curr_freq
    normed_words[curr_tok] = np.log(abs(curr_score)/words_sum) - 4/np.log(1+curr_freq)
    original_score[curr_tok] = curr_score

  import operator
  sorted_words = sorted(normed_words.items(), key=operator.itemgetter(1), reverse=True)

  # Token, Importance score, Frequency, Original logistic regression score
  for item in sorted_words[:50]:
      print("{: <16}".format(item[0]), "\t%.6f"%(item[1]),"\t%5d"%(frequency_dict[item[0]]),"\t%.6f"%(original_score[item[0]])) # "\t{}".format("\t".join([str(item) for item in atten_words[item[0]]])))

worst            	-8.163862 	  270 	4.076367
lacks            	-8.229741 	  134 	4.223908
lacking          	-8.240253 	  106 	4.352772
remarkable       	-8.333685 	  176 	-3.647869
mess             	-8.362296 	  149 	3.636636
powerful         	-8.383788 	  242 	-3.318314
stupid           	-8.399924 	  142 	3.529367
failure          	-8.423783 	   83 	3.796342
beautiful        	-8.428619 	  251 	-3.157676
fascinating      	-8.455213 	  304 	-3.001472
wonderful        	-8.459947 	  174 	-3.220645
hilarious        	-8.462791 	  192 	-3.165558
bad              	-8.465366 	  886 	2.661729
enjoyable        	-8.471997 	  266 	-3.001078
devoid           	-8.502347 	   73 	3.604056
none             	-8.514476 	  130 	3.193304
lack             	-8.528424 	  166 	3.028858
waste            	-8.534680 	  114 	3.200727
refreshing       	-8.541350 	   77 	-3.427487
charming         	-8.555369 	  232 	-2.810856
terrific         	-8.562534 	  156 	-2.955358
solid            	-8.571638 	  233 	-2.763903

In [ ]:
print(frequency_dict['is'])

8115


In [ ]:
if dataset == "sst2":
  token = 'downer'
else:
  token = "inattentive"
print(frequency_dict[token])
print(atten_words[token])

# you can have negative attention? Is this like the wang cutello paper where negative indicates bad sentiment and positive indicates good sentiment?
print(abs(sum(atten_words[token]))/len(atten_words[token]))

9
[-0.23070532, -0.20780681, -0.1133861]
0.18396607786417007


In [ ]:
a = [item[0] for item in sorted_words[:50]]
print(", ".join(a))

worst, lacks, lacking, remarkable, mess, powerful, stupid, failure, beautiful, fascinating, wonderful, hilarious, bad, enjoyable, devoid, none, lack, waste, refreshing, charming, terrific, solid, too, pointless, poorly, depressing, appealing, clichés, best, bore, less, pretentious, neither, dull, unfunny, flat, poor, treat, entertaining, fails, ugly, engrossing, heart, unnecessary, touching, no, boring, brilliant, intriguing, hardly


### Dump normalized frequency and attention scores

In [ ]:
normed_freq = dict()
fre_sum = sum(list(frequency_dict.values())) # num unique tokens?
data = [elem for elem in list(frequency_dict.values())]
fre_mean = np.mean(data)
fre_max = np.max(data)
fre_std = np.std(data)

print("unique tokens, average frequency, freq std dev and freq max")
print(fre_sum, fre_mean, fre_std, fre_max)
for k in frequency_dict:
    normed_freq[k] = (frequency_dict[k]/fre_sum)# / fre_std
print("frequency of word \'lacks\' and normed frequency of word \'lacks\'")
print(frequency_dict['lacks'], normed_freq['lacks'])


if not use_logistic_regression:    
  new_words = dict()
  for token in atten_words:
      if len(atten_words[token]) > 0:
          new_words[token] = abs(sum(atten_words[token]))/len(atten_words[token])
 
normed_words = dict()
words_sum = sum(new_words.values())
data = [elem for elem in list(new_words.values())]
atten_max = np.max(data)
atten_mean = np.mean(data)
atten_std = np.std(data)
print("words sum, attention mean, attention std dev")
print(words_sum, atten_mean, atten_std)
for k in new_words:
    normed_words[k] = (new_words[k]/words_sum)# / atten_std

print(new_words.keys())
print("average attention for word \'lacks\' and normalized average attention for word \'lacks\'")
print(new_words['lacks'], normed_words['lacks'])
print("average attention for word \'complaints\' and normalized average attention for word \'complaints\'")
print(new_words['terrible'], normed_words['terrible'])

unique tokens, average frequency, freq std dev and freq max
527590 49.78673209398887 381.4068814422797 20041
frequency of word 'lacks' and normed frequency of word 'lacks'
134 0.0002539851020678936
words sum, attention mean, attention std dev
7009.719217154006 0.6614814775081633 0.5363650239263338
dict_keys(['lacking', 'lacks', 'worst', 'failure', 'mess', 'devoid', 'stupid', 'bore', 'squanders', 'waste', 'none', 'depressing', 'pointless', 'nowhere', 'loses', 'lack', 'poorly', 'lousy', 'unnecessary', 'poor', 'neither', 'hardly', 'lackluster', 'listless', 'clichés', 'unfunny', 'pretentious', 'wannabe', 'useless', 'repetitive', 'distasteful', 'copy', 'tiresome', 'unfortunately', 'pretends', 'generic', 'lame', 'bad', 'ugly', 'fails', 'insulting', 'uninspired', 'flat', 'insult', 'unpleasant', 'irritating', 'uneven', 'charmless', 'amateurish', 'cliché', 'shame', 'boring', 'obnoxious', 'scarcely', 'worse', 'elsewhere', 'mistake', 'weak', 'dull', 'overbearing', 'muddled', 'unimaginative', 'dis

In [ ]:
import pickle

saved_words = {}

for k in normed_words:
    saved_words[k] = {'normed_atten': normed_words[k], 'normed_fre': normed_freq[k]}

if dataset == "sst2":
  path = "logistic_regression/glue_data/SST-2/"
  if not os.path.exists(path):
    os.makedirs(path)
  pickle.dump(saved_words, open(path + "saved_words.p", 'wb'))
else:
  path = "logistic_regression/yelp_data/"
  if not os.path.exists(path):
    os.makedirs(path)
  pickle.dump(saved_words, open(path + "saved_words.p", 'wb'))


### Compare two lists

In [ ]:
# words_sst = pickle.load(open("transformers/glue_data/SST-2/saved_words.p", "rb"))
# words_yelp = pickle.load(open("transformers/yelp_data/saved_words.p", "rb"))
words_sst = pickle.load(open("logistic_regression/glue_data/SST-2/saved_words.p", "rb"))
words_yelp = pickle.load(open("logistic_regression/yelp_data/saved_words.p", "rb"))

In [ ]:
vocab_union = set(words_sst.keys()).union(set(words_yelp.keys()))
# print(len(set(words_sst.keys()).intersection(set(words_yelp.keys())))) # Intersection contains 2686 words, need to cross reference the paper tomorrow
print(len(vocab_union))
id2word_union = {}
word2id_union = {}
for idx, w in enumerate(vocab_union):
    id2word_union[idx] = w
    word2id_union[w] = idx

22582


In [ ]:
print(len(words_sst), len(words_yelp))

10597 18183


In [ ]:
# Original fre_sum
# total_fre_sst = 768543
# total_fre_yelp = 363504

# Our attention fre_sum
# total_fre_sst = 18849 
# total_fre_yelp = 753242 

# Our Logistic Regression fre_sum
total_fre_sst = 527590  
total_fre_yelp = 3032301 

In [ ]:
sst_all = dict()
yelp_all = dict()
for i in range(len(id2word)):
    w = id2word[i]
    if w in words_sst:
        tmp = np.log(words_sst[w]['normed_atten']) - 4/np.log(1+words_sst[w]['normed_fre'] * total_fre_sst)
        sst_all[w] = tmp
    
    if w in words_yelp:
        tmp = np.log(words_yelp[w]['normed_atten']) - 4/np.log(1+words_yelp[w]['normed_fre'] * total_fre_yelp)
        yelp_all[w] = tmp
        
sst_max, sst_min = max(sst_all.values()), min(sst_all.values())
yelp_max, yelp_min = max(yelp_all.values()), min(yelp_all.values())
max_sst_fre = max([words_sst[w]['normed_fre'] for w in words_sst])
max_yelp_fre = max([words_yelp[w]['normed_fre'] for w in words_yelp])
diff_all = dict()
for w in word2id:
    if w in words_sst:
        sst_v = (sst_all[w] - sst_min) / (sst_max - sst_min)
    
        if w in words_yelp:
            yelp_v = (yelp_all[w] - yelp_min) / (yelp_max - yelp_min)
        else:
            yelp_v = 0
        
        diff_all[w] = (sst_v - yelp_v)
        
# exclude "good"
sorted_words = sorted(diff_all.items(), key=operator.itemgetter(1), reverse=True)
sst_values = list(sst_all.values())
yelp_values = list(yelp_all.values())
sst_values.sort(reverse=True)
yelp_values.sort(reverse=True)

print("{: <16}\t{}\t{}\t{}\t{}\t{}\t{}\t{}".format("**word**", "diff", "s score", "y score", "s rank",
                                                  "s fre", "y rank", "y fre"))

atten_sum = 0
for item in sorted_words[:50]:
    w = item[0]
    idx = word2id[w]
    atten_sum += (sst_all[w] - sst_min) / (sst_max - sst_min)
    print("{: <16}\t{:2.4f}\t{:02.2f}\t{:2.2f}\t{:05}\t{:5.2f}\t{:05}\t{:5.2f}".format(w, diff_all[w],
                            (sst_all[w] - sst_min) / (sst_max - sst_min) if w in words_sst else 0, 
                            (yelp_all[w] - yelp_min) / (yelp_max - yelp_min) if w in words_yelp else 0,                                                     
                             sst_values.index(sst_all[w]) if w in words_sst else 0, 
                              (words_sst[w]['normed_fre'] if w in words_sst else 0) * total_fre_sst,
                              yelp_values.index(yelp_all[w]) if w in words_yelp else 0, 
                              (words_yelp[w]['normed_fre'] if w in words_yelp else 0)* total_fre_yelp
                             ))
#     print("{: <16}\t{:5.2f}".format(w,
#                               (words_sst[w]['normed_fre'] if w in words_sst else 0) * total_fre_sst))
print(atten_sum/50)


**word**        	diff	s score	y score	s rank	s fre	y rank	y fre
provocative     	0.9446	0.94	0.00	00070	103.00	00000	 0.00
compelling      	0.9444	0.94	0.00	00073	356.00	00000	 0.00
superficial     	0.9428	0.94	0.00	00080	73.00	00000	 0.00
captures        	0.9410	0.94	0.00	00088	91.00	00000	 0.00
violence        	0.9389	0.94	0.00	00097	130.00	00000	 0.00
dazzling        	0.9322	0.93	0.00	00133	65.00	00000	 0.00
succeeds        	0.9313	0.93	0.00	00140	71.00	00000	 0.00
clarity         	0.9287	0.93	0.00	00150	80.00	00000	 0.00
affection       	0.9210	0.92	0.00	00191	86.00	00000	 0.00
sensual         	0.9162	0.92	0.00	00230	48.00	00000	 0.00
vivid           	0.9148	0.91	0.00	00245	83.00	00000	 0.00
embracing       	0.9085	0.91	0.00	00293	20.00	00000	 0.00
fatal           	0.9084	0.91	0.00	00295	60.00	00000	 0.00
anatomical      	0.9070	0.91	0.00	00308	17.00	00000	 0.00
prose           	0.9055	0.91	0.00	00322	14.00	00000	 0.00
wry             	0.9028	0.90	0.00	00347	47.00	00000	 0.00
sitco

In [ ]:
print([item[0] for item in sorted_words[:300]])

['provocative', 'compelling', 'superficial', 'captures', 'violence', 'dazzling', 'succeeds', 'clarity', 'affection', 'sensual', 'vivid', 'embracing', 'fatal', 'anatomical', 'prose', 'wry', 'sitcom', 'reveals', 'spielberg', 'scarcely', 'meaningful', 'bleak', 'explores', 'dickens', 'absorbing', 'startling', 'exploitation', 'flawed', 'detached', 'collapses', 'comedies', 'uniquely', 'achievement', 'haunting', 'irresistible', 'triumph', 'distinguished', 'chord', 'bracing', 'forceful', 'poetic', 'radical', 'gripping', 'bourne', 'transforms', 'ponder', 'sweetly', 'deaths', 'earnest', 'whimper', 'stereotypes', 'friendship', 'dynamics', 'breakthrough', 'flatly', 'subtly', 'bile', 'warriors', 'stunt', 'expressive', 'invasion', 'fragmented', 'poem', 'assumes', 'imagery', 'bard', 'pollution', 'uncertain', 'mysteries', 'hypnotic', 'illuminating', 'credible', 'propaganda', 'thriller', 'historically', 'sly', 'richly', 'argentine', 'adaptation', 'contradictory', 'emptiness', 'experimental', 'lawrence'

In [ ]:
potential_shortcuts = [item[0] for item in sorted_words[:300]]
shortcuts2cnt = dict()

# Go through each sentence of the dataset, extract tokens
for idx in range(len(preds_labels)):
    sen = sentences[idx]
    tokens = [id2word[item] for item in tokenizer(sen)["input_ids"]] # is the number of tokens the same as the number of words in the sentence
    tokens = combine_tokens(tokens)
    for w in potential_shortcuts:
        if w in tokens:
            if w not in shortcuts2cnt:
                shortcuts2cnt[w] = []
            tmp = shortcuts2cnt[w]
            tmp.append(idx)
            shortcuts2cnt[w] = tmp
for w in shortcuts2cnt:
    print(w, len(shortcuts2cnt[w]))

superficial 1
succeeds 2
feminist 2
conspiracy 2
horrors 1
vivid 3
emptiness 1
relentless 1
eastwood 1
earnest 4
historically 1
extremes 1
cinematic 3
storytelling 1
tissues 1
bourne 1
conviction 4
collapse 1
portrayal 2
reveals 2
adaptation 2
sequel 2
compelling 4
unpredictable 2
conclusions 1
ponder 1
flashes 1
subordinate 2
crossover 1
blair 2
esther 1
grips 1
viewers 2
haunting 3
momentum 1
filmmaker 1
backward 2
argentine 1
celebrates 1
vulnerable 1
violence 2
thriller 4
seeks 1
thematic 2
captures 1
submarine 1
sitcom 1
wry 1
plots 1
plotted 1
predecessor 2
comedies 3
meaningful 1
literary 1
bard 1
bleak 1
poem 1
affection 1
pathology 1
embracing 1
intellectual 1
stereotypes 1
comedic 1
restoring 1
detached 1
exaggerated 1
yarn 1
innocence 1
kwan 1
spielberg 1
stylistic 1


In [ ]:
import random

for w in shortcuts2cnt:
    print(w)
    tmp = shortcuts2cnt[w]
    random.shuffle(tmp)
    for idx in tmp[:5]:
        print(sentences[idx])
    print("")

bleak
unflinchingly bleak and desperate 

filmmaker
not since japanese filmmaker akira kurosawa 's ran have the savagery of combat and the specter of death been visualized with such operatic grandeur . 
ramsay , as in ratcatcher , remains a filmmaker with an acid viewpoint and a real gift for teasing chilly poetry out of lives and settings that might otherwise seem drab and sordid . 
displaying about equal amounts of naiveté , passion and talent , beneath clouds establishes sen as a filmmaker of considerable potential . 
allows us to hope that nolan is poised to embark a major career as a commercial yet inventive filmmaker . 

violence
this re-do is so dumb and so exploitative in its violence that , ironically , it becomes everything that the rather clumsy original was railing against . 
... mafia , rap stars and hood rats butt their ugly heads in a regurgitation of cinematic violence that gives brutal birth to an unlikely , but likable , hero . ' 
excessive , profane , packed with car

In [ ]:
# I think there is supposed to be 8500 shortcuts, but we are now only getting 3
shortcut_sentences = set()
for k in shortcuts2cnt:
    for s_id in shortcuts2cnt[k]:
        shortcut_sentences.add(s_id)
print(len(shortcut_sentences))
shortcut_sentences = list(shortcut_sentences)

381


## Step 3

In [ ]:
syns = pickle.load(open( "sst_syns.p", "rb" )) # load synonyms

generated_sens = dict()
generated_labels = dict()

for w in shortcuts2cnt:
    tmp = []
    
    if w in syns:
        replace_words = syns[w]
    else:
        replace_words = syns[w.lower()]
    
    for sen in shortcuts2cnt[w]:
        generated_labels[w] = labels[sen]
        sen = sentences[sen]
        
        if " "+w+" " in sen:
            for t in replace_words:
                tmp.append(sen.replace(" "+w+" ", " "+t+" "))
        elif w+" " in sen:
            for t in replace_words:
                tmp.append(sen.replace(w+" ", t+" "))
        elif " "+w in sen:
            for t in replace_words:
                tmp.append(sen.replace(" "+w, " "+t))
        elif w in sen:
            for t in replace_words:
                tmp.append(sen.replace(w, t))
        else:
            print("error")
            print(w, sen)
    generated_sens[w] = tmp

In [ ]:
generated_sens_list = []
generated_labels_list = []
for k in generated_sens.keys():
    generated_sens_list += generated_sens[k]
    generated_labels_list += [generated_labels[k]] * len(generated_sens[k])
print(len(generated_sens_list))
print(len(generated_labels_list))

import csv

with open('./perturb/dev.tsv', 'wt') as out_file:
    tsv_writer = csv.writer(out_file, delimiter='\t')
    tsv_writer.writerow(['sentence', "label"])
    for idx, sentence in enumerate(generated_sens_list):
        tsv_writer.writerow([sentence.strip().replace("\n", " "), generated_labels_list[idx]])
    print(idx)

93405
93405
93404


In [ ]:
# eval the generated sentences using the trained model

In [ ]:
ori_acc = dict()
for k in generated_sens.keys():
    ids = shortcuts2cnt[k]
    acc = np.sum(np.equal(np.array(preds_labels)[ids], np.array(labels)[ids]))/len(ids)
    ori_acc[k] = acc
print(ori_acc)

generated_preds = np.load("./perturb/predictions.npy")

generated_acc = dict()
pre_idx = 0
for k in generated_sens.keys():
    tmp_len = len(generated_sens[k])
    cur_labels = generated_labels_list[pre_idx:pre_idx+tmp_len]
    cur_preds = generated_preds[pre_idx:pre_idx+tmp_len]
    cur_preds_labels = []
    for elem in cur_preds:
        if elem[0] > elem[1]:
            cur_preds_labels.append(0)
        else:
            cur_preds_labels.append(1)
#     print(tmp_len, len(labels), len(preds_labels), len(preds))
    acc = np.sum(np.equal(np.array(cur_preds_labels), np.array(cur_labels)))/len(cur_preds_labels)
    generated_acc[k] = acc
    pre_idx += tmp_len
print(generated_acc)

{'superficial': 1.0, 'depressed': 1.0, 'redundant': 1.0, 'succeeds': 1.0, 'crude': 1.0, 'oblivious': 1.0, 'sharply': 1.0, 'recycled': 1.0, 'proves': 0.9858156028368794, 'vivid': 1.0, 'pointless': 1.0, 'touching': 1.0, 'emptiness': 1.0, 'woven': 1.0, 'cruel': 1.0, 'historically': 1.0, 'wisdom': 1.0, 'fluid': 1.0, 'seal': 1.0, 'casts': 1.0, 'masterpiece': 1.0, 'hopeless': 1.0, 'clumsy': 1.0, 'conviction': 0.9846153846153847, 'realistic': 1.0, 'portrayal': 1.0, 'reveals': 1.0, 'uneven': 0.9880952380952381, 'unpredictable': 1.0, 'epic': 1.0, 'considerable': 1.0, 'enables': 1.0, 'flashes': 1.0, 'vicious': 1.0, 'jagged': 1.0, 'slick': 1.0, 'essentially': 1.0, 'haunting': 1.0, 'pale': 1.0, 'affecting': 1.0, 'intelligent': 0.9914893617021276, 'backward': 1.0, 'thrill': 1.0, 'powerful': 1.0, 'torture': 1.0, 'celebrates': 1.0, 'strongest': 1.0, 'irrelevant': 1.0, 'canvas': 1.0, 'smug': 1.0, 'captures': 1.0, 'offensive': 1.0, 'tale': 0.9943342776203966, 'sitcom': 1.0, 'abuse': 1.0, 'suffers': 1.0

In [ ]:
acc_diff = dict()
for k in ori_acc:
    acc_diff[k] = abs(ori_acc[k] - generated_acc[k])
    

for idx, (k, v) in enumerate(acc_diff.items()):
    if v < 0.3:
        continue
    print("{}\t{:.2f}\t{:.2f}\t{:.2f}\t{}".format(k, v, ori_acc[k], 
                                                  generated_acc[k], ",".join(syns[k][:3])))

oblivious	0.32	1.00	0.68	insensitive,uncaring,unsympathetic
recycled	0.89	1.00	0.11	recycle,recycling,recycles
cruel	0.31	1.00	0.69	brutal,brutish,ruthless
seal	0.40	1.00	0.60	sealed,hermetic,stamped
casts	0.96	1.00	0.04	castings,throws,puts
slick	0.77	1.00	0.23	blot,blemish,spot
pale	0.45	1.00	0.55	bali,faint,livid
sitcom	0.43	1.00	0.57	comic,sitcoms,comedian
juvenile	0.81	1.00	0.19	juveniles,underage,minors
affection	0.41	1.00	0.59	tenderness,fondness,ailment
capable	0.60	1.00	0.40	able,ability,capacity
deadly	0.89	1.00	0.11	mortal,fatal,lethal
dramatically	0.87	1.00	0.13	drastically,markedly,significantly
longest	0.84	1.00	0.16	tallest,longer,most
fallen	0.58	1.00	0.42	declined,fell,shrunk
qualities	0.32	1.00	0.68	qualifications,qualification,attributes
glossy	0.68	0.95	0.28	shiny,lustrous,gloss
rises	0.49	0.97	0.48	rise,soars,soar
graceful	0.45	1.00	0.55	elegant,tasteful,courtesy
problematic	0.44	0.97	0.53	difficult,tricky,challenging
harmless	0.65	1.00	0.35	inoffensive,innocuous,b